In [ ]:
import os, sys, warnings
from pathlib import Path
_r = Path.cwd()
while not (_r / 'requirements.txt').exists() and _r != _r.parent:
    _r = _r.parent
os.chdir(_r); sys.path.insert(0, str(_r / 'src')); warnings.filterwarnings('ignore')

# Same km driven => same degradation?  (CLEAN odometer, ORIGINAL data)

For vehicles whose **odometer is recorded from ~0 km while still at 100% SoH** (so the km axis is real, not a
sentinel artefact), we plot SoH vs distance straight from the raw telemetry and read SoH off at a shared, long km.

- **Euler**: per-reading capacity SoH = remainingCapacity / (SoC/100) at SoC 95-100%, daily medians, raw odometer.
- **Mahindra**: **omitted** - its odometer is only logged *after* heavy use (every vehicle's first reading is
  already tens of thousands of km at a degraded SoH), so a fresh-baseline km axis is impossible there.

In [ ]:
import numpy as np, pandas as pd
import plotly.graph_objects as go, plotly.express as px
PAL = px.colors.qualitative.Dark24

def euler_daily(vin):                              # raw capacity SoH -> daily, CLEAN odometer only
    d = pd.read_parquet(f'data/euler/dense/{vin}.parquet')
    soc = pd.to_numeric(d['batterySoc'], errors='coerce'); rc = pd.to_numeric(d['batteryRemainingCapacity'], errors='coerce')
    odo = pd.to_numeric(d['odometer'], errors='coerce'); t = pd.to_datetime(d['t'])
    m = soc.between(95, 100) & rc.between(0, 200) & odo.between(1, 150000)
    df = pd.DataFrame({'day': t[m].dt.normalize(), 'odo': odo[m], 'fc': rc[m] / (soc[m] / 100.0)})
    if len(df) < 40: return None
    med = df['fc'].median(); df = df[df['fc'].between(0.6 * med, 1.4 * med)]
    g = df.groupby('day').agg(odo=('odo', 'max'), fc=('fc', 'median')).reset_index().sort_values('day')
    g['odo'] = g['odo'].cummax()
    if g['odo'].iloc[0] >= 3000: return None         # odometer must start near 0 km (fresh) -> trustworthy km axis
    g['km'] = g['odo'] - g['odo'].iloc[0]
    base = g['fc'].head(20).quantile(0.9); g['soh'] = (100 * g['fc'] / base).clip(upper=100)
    return g if g['km'].max() >= 5000 else None

def fkm(g):
    odo = pd.to_numeric(g['odo_max'], errors='coerce').cummax()
    return float(odo.diff().clip(lower=0, upper=8000).fillna(0).sum())

# Euler: longest-distance vehicles with a CLEAN odometer (recorded from ~0 km) that START at 100% SoH
fe = pd.read_parquet('data/euler/features/feature_table.parquet').sort_values(['vin', 'month'])
etot = pd.Series({v: fkm(g) for v, g in fe.groupby('vin')})
_fst = fe.groupby('vin')['soh'].first()
_cand = etot[_fst >= 99.9].sort_values(ascending=False).index.tolist()[:40]
_loaded = {}
for v in _cand:
    g = euler_daily(v)
    if g is not None and float(g['soh'].iloc[0]) >= 99.5:
        _loaded[v] = g
    if len(_loaded) >= 6: break
_mx = pd.Series({v: g['km'].max() for v, g in _loaded.items()}).sort_values(ascending=False)
EU = {v: _loaded[v] for v in _mx.index[:5]}
EU_KM = int(min(g['km'].max() for g in EU.values()))
print('Euler (clean odometer, 100% start): ' + ', '.join(f'{v[-6:]} {int(g["km"].max())}km' for v, g in EU.items()) + f'  | compare at {EU_KM} km')

## Euler — raw capacity SoH

In [ ]:
# Euler - vehicles matched on distance: do their SoH coincide at the same km?
fig = go.Figure()
for i, (v, g) in enumerate(EU.items()):
    fig.add_scatter(x=g['km'], y=g['soh'], mode='lines+markers', line=dict(color=PAL[i], width=1.4), marker=dict(size=3, color=PAL[i]),
                    name=v[-6:], opacity=0.85, hovertemplate=v[-6:] + '  %{x:.0f} km  %{y:.1f}%<extra></extra>')
fig.add_vline(x=EU_KM, line=dict(color='gray', width=1.3, dash='dash'), annotation_text=f'{EU_KM} km', annotation_position='top')
fig.update_layout(height=520, width=1050, template='plotly_white',
                  title='Euler - 5 vehicles with ~the same distance driven: raw SoH vs km',
                  xaxis_title='km driven (raw odometer)', yaxis_title='SoH (%) [raw]', yaxis_range=[70, 101])
fig.show()
vals = {v[-6:]: float(np.interp(EU_KM, g['km'], g['soh'])) for v, g in EU.items()}
print(f'SoH at {EU_KM} km (same distance): ' + ', '.join(f'{k} {x:.0f}%' for k, x in vals.items()) + f'   -> spread {max(vals.values())-min(vals.values()):.0f} pp')

## Euler — raw vs **derived** SoH at the same distance

Left = *raw* per-reading capacity SoH (daily medians, noisy). Right = *derived* SoH — the validated
BMS-capacity method the model trains on (high-SoC full-capacity, **isotonic** monotone fit, per-vehicle
nominal). Same vehicles, same colours, same km checkpoint. The derived curve removes the wiggle/recovery
artefacts of the raw signal while preserving the same-distance spread.

In [ ]:
from plotly.subplots import make_subplots

def eu_derived_series(vin):                                  # validated isotonic monthly SoH vs km
    g = fe[fe['vin'] == vin].sort_values('month')
    odo = pd.to_numeric(g['odo_max'], errors='coerce').cummax()
    km = odo.diff().clip(lower=0, upper=8000).fillna(0).cumsum()
    return km.to_numpy(), g['soh'].to_numpy()

fig = make_subplots(rows=1, cols=2, shared_yaxes=True, horizontal_spacing=0.05,
                    subplot_titles=('raw capacity SoH (daily, noisy)', 'derived SoH (validated, isotonic monthly)'))
for i, (v, g) in enumerate(EU.items()):
    t = v[-6:]
    fig.add_scatter(x=g['km'], y=g['soh'], mode='lines+markers', line=dict(color=PAL[i], width=1.3),
                    marker=dict(size=3, color=PAL[i]), name=t, legendgroup=t, opacity=0.85, row=1, col=1,
                    hovertemplate=t + ' raw  %{x:.0f} km  %{y:.1f}%<extra></extra>')
    dk, ds = eu_derived_series(v)
    fig.add_scatter(x=dk, y=ds, mode='lines+markers', line=dict(color=PAL[i], width=2.0),
                    marker=dict(size=4, color=PAL[i]), name=t, legendgroup=t, showlegend=False, row=1, col=2,
                    hovertemplate=t + ' derived  %{x:.0f} km  %{y:.1f}%<extra></extra>')
for cc in (1, 2):
    fig.add_vline(x=EU_KM, line=dict(color='gray', width=1.1, dash='dash'), row=1, col=cc)
fig.update_xaxes(title_text='km driven (raw odometer)', row=1, col=1)
fig.update_xaxes(title_text='km driven (raw odometer)', row=1, col=2)
fig.update_yaxes(title_text='SoH (%)', range=[70, 101], row=1, col=1)
fig.update_layout(height=520, width=1180, template='plotly_white',
                  title=f'Euler - same-distance vehicles: raw vs derived SoH  (checkpoint {EU_KM} km)',
                  legend=dict(title='vehicle', x=1.01, y=1))
fig.show()
dvals = {}
for v, g in EU.items():
    dk, ds = eu_derived_series(v)
    dvals[v[-6:]] = float(np.interp(EU_KM, dk, ds))
print(f'derived SoH at {EU_KM} km (same distance): ' + ', '.join(f'{k} {x:.0f}%' for k, x in dvals.items())
      + f'   -> spread {max(dvals.values()) - min(dvals.values()):.0f} pp')

## Euler — what separates the heavy degrader from the flat ones? (top-5 stress features)

Ranked three independent ways (XGB gain · monthly-loss Spearman · cross-vehicle discrimination) and **verified to
order the heavy degrader 217054 above the flat 217278**. The five operational drivers that survive: **mean pack
temperature**, **depth-of-discharge**, **discharge-current magnitude**, **high-SoC dwell**, **cell imbalance** —
each rises from the flat 217278 toward the heavy 217054 / 217044.

**Honest caveat:** in this cohort *no single operational stressor is a strong discriminator* (best cross-vehicle
|ρ| ≈ 0.28); per-vehicle SoH drop is dominated by age/distance. The obvious usage proxies (Ah-throughput, C-rate)
were **excluded** because the flat 217278 is a *young, high-utilization* vehicle that simply hasn't aged yet — so
throughput points the wrong way. `dod_mean` (~76%) and `imbalance_mean` (~66%, NaN on the older 2022 cohort) have
partial coverage; gaps are not interpolated.

In [ ]:
from plotly.subplots import make_subplots

# verified top-5 operational degradation features (col, panel title, y-label, take_magnitude)
FEATS5 = [('temp_mean',     'Mean pack temperature',        'temp (°C)', False),
          ('dod_mean',      'Mean depth-of-discharge',      'DoD (%)',      False),
          ('cur_dis_mean',  'Discharge-current magnitude',  '|A|',          True),
          ('frac_soc_high', 'Fraction of time at high SoC', 'hi-SoC',       False),
          ('imbalance_mean','Mean cell-voltage imbalance',  'imbal (mV)',   False)]

def feat_frame(vin):
    g = fe[fe['vin'] == vin].sort_values('month').copy()
    odo = pd.to_numeric(g['odo_max'], errors='coerce').cummax()
    g['km'] = odo.diff().clip(lower=0, upper=8000).fillna(0).cumsum()       # same km axis as the SoH-vs-km plot
    return g

VORDER = list(EU.keys())                                                     # [217054,217135,217125,217044,217278] -> PAL[0..4]
FR = {v: feat_frame(v) for v in VORDER}

fig = make_subplots(rows=5, cols=1, shared_xaxes=True, vertical_spacing=0.035,
                    subplot_titles=[t for _, t, _, _ in FEATS5])
for r, (col, _, ylab, mag) in enumerate(FEATS5, start=1):
    for i, v in enumerate(VORDER):
        g = FR[v]; t = v[-6:]
        y = pd.to_numeric(g[col], errors='coerce').abs() if mag else pd.to_numeric(g[col], errors='coerce')
        ys = y.rolling(3, center=True, min_periods=1).mean().where(y.notna())   # light 3-mo smoother, keep NaN gaps
        fig.add_scatter(x=g['km'], y=y, mode='markers', marker=dict(size=3, color=PAL[i]), opacity=0.30,
                        legendgroup=t, showlegend=False, row=r, col=1, connectgaps=False,
                        hovertemplate=t + '  %{x:.0f} km  %{y:.2f}<extra></extra>')
        fig.add_scatter(x=g['km'], y=ys, mode='lines', line=dict(color=PAL[i], width=1.9), name=t,
                        legendgroup=t, showlegend=(r == 1), row=r, col=1, connectgaps=False,
                        hovertemplate=t + ' (3-mo avg) %{y:.2f}<extra></extra>')
    fig.add_vline(x=EU_KM, line=dict(color='gray', width=1, dash='dash'), row=r, col=1)
    fig.update_yaxes(title_text=ylab, row=r, col=1)
fig.update_xaxes(title_text='km driven (raw odometer)', row=5, col=1)
fig.update_layout(height=1180, width=1050, template='plotly_white',
                  title='Euler - top-5 degradation-stressor trends vs distance (same-distance vehicles; discharge current shown as magnitude)',
                  legend=dict(title='vehicle', orientation='h', y=1.045, x=0, font=dict(size=10)))
fig.show()

# per-vehicle history means vs total SoH drop -> the gradient the noisy traces can hide
rows = []
for v in VORDER:
    g = fe[fe['vin'] == v].sort_values('month')
    rec = {'vin': v[-6:], 'total_drop_pp': round(float(g['soh'].iloc[0] - g['soh'].iloc[-1]), 2)}
    for col, _, _, mag in FEATS5:
        s = pd.to_numeric(g[col], errors='coerce'); s = s.abs() if mag else s
        rec[col] = round(float(s.mean()), 2)
    rows.append(rec)
summ = pd.DataFrame(rows).sort_values('total_drop_pp').reset_index(drop=True)
print('Per-vehicle mean stress vs total SoH drop (flat -> heavy):')
print(summ.to_string(index=False))

### Two-vehicle deep-dive: 217054 (heavy, −8.7 pp) vs 217135 (−3.4 pp)

Each panel = one stress feature (**solid line, left axis**) with that vehicle's **SoH overlaid (dotted line,
right axis)**, so you can see where SoH bends relative to the stressor. Same two colours as the SoH-vs-km plot.

In [ ]:
# two-vehicle feature comparison with SoH overlaid (uses FR, FEATS5 from the eu_feats cell)
tail2full = {v[-6:]: v for v in EU.keys()}
VTWO = [tail2full['217054'], tail2full['217135']]
CMAP = {'217054': PAL[0], '217135': PAL[1]}
_soh_min = min(float(FR[v]['soh'].min()) for v in VTWO)

fig = make_subplots(rows=5, cols=1, shared_xaxes=True, vertical_spacing=0.04,
                    subplot_titles=[t for _, t, _, _ in FEATS5],
                    specs=[[{'secondary_y': True}] for _ in range(5)])
for r, (col, _, ylab, mag) in enumerate(FEATS5, start=1):
    for v in VTWO:
        g = FR[v]; t = v[-6:]; c = CMAP[t]
        y = pd.to_numeric(g[col], errors='coerce').abs() if mag else pd.to_numeric(g[col], errors='coerce')
        ys = y.rolling(3, center=True, min_periods=1).mean().where(y.notna())
        fig.add_scatter(x=g['km'], y=y, mode='markers', marker=dict(size=3, color=c), opacity=0.22,
                        legendgroup=t, showlegend=False, row=r, col=1, secondary_y=False, connectgaps=False,
                        hovertemplate=t + ' ' + col + ' %{y:.2f}<extra></extra>')
        fig.add_scatter(x=g['km'], y=ys, mode='lines', line=dict(color=c, width=2.4), name=f'{t} · stress',
                        legendgroup=t, showlegend=(r == 1), row=r, col=1, secondary_y=False, connectgaps=False)
        fig.add_scatter(x=g['km'], y=g['soh'], mode='lines', line=dict(color=c, width=1.5, dash='dot'),
                        name=f'{t} · SoH', legendgroup=t + 'soh', showlegend=(r == 1), opacity=0.75,
                        row=r, col=1, secondary_y=True, hovertemplate=t + ' SoH %{y:.1f}%<extra></extra>')
    fig.update_yaxes(title_text=ylab, row=r, col=1, secondary_y=False)
    fig.update_yaxes(title_text='SoH %', range=[_soh_min - 1, 101], showgrid=False, color='gray',
                     row=r, col=1, secondary_y=True)
    fig.add_vline(x=EU_KM, line=dict(color='gray', width=1, dash='dash'), row=r, col=1)
fig.update_xaxes(title_text='km driven (raw odometer)', row=5, col=1)
fig.update_layout(height=1260, width=1050, template='plotly_white',
                  title='Euler 217054 (heavy) vs 217135: top-5 stress features (solid, left axis) + SoH overlaid (dotted, right axis)',
                  legend=dict(orientation='h', y=1.035, x=0, font=dict(size=9)))
fig.show()

print('History-mean stress — 217054 (heavy, -8.7pp) vs 217135 (-3.4pp):')
rows = []
for v in VTWO:
    g = fe[fe['vin'] == v].sort_values('month')
    rec = {'vin': v[-6:], 'total_drop_pp': round(float(g['soh'].iloc[0] - g['soh'].iloc[-1]), 2)}
    for col, _, _, mag in FEATS5:
        s = pd.to_numeric(g[col], errors='coerce'); s = s.abs() if mag else s
        rec[col] = round(float(s.mean()), 2)
    rows.append(rec)
print(pd.DataFrame(rows).to_string(index=False))

### Same distance, different degradation — why?

217054 and 217135 drove ~the same distance, yet 217054 lost **8.7 pp** vs **3.4 pp**. Top panel = the outcome
(SoH vs km); the two panels below are the only stressors that separate them in the right direction — **217054
draws heavier discharge current and cycles deeper**. Dotted lines = each vehicle's history mean.

In [ ]:
# Same distance, different degradation: 217054 vs 217135, TRUNCATED to the common distance both reach
tail2full = {v[-6:]: v for v in EU.keys()}
VTWO = [tail2full['217054'], tail2full['217135']]
CMAP = {'217054': PAL[0], '217135': PAL[1]}
EXPL = [('cur_dis_mean', 'current |A|', True), ('dod_mean', 'DoD %', False)]

CHK = int(min(FR[v]['km'].max() for v in VTWO))            # common distance -> compare strictly same-distance
FT = {v: FR[v][FR[v]['km'] <= CHK] for v in VTWO}          # validated SoH + features, up to CHK
RT = {v: EU[v][EU[v]['km'] <= CHK] for v in VTWO}          # raw daily SoH, up to CHK

def fmean(v, col, mag):
    s = pd.to_numeric(FT[v][col], errors='coerce'); return float((s.abs() if mag else s).mean())
drop = {v: float(FT[v]['soh'].iloc[0] - FT[v]['soh'].iloc[-1]) for v in VTWO}
amean = {v: {col: fmean(v, col, mag) for col, _, mag in EXPL} for v in VTWO}
_pct = (amean[VTWO[0]]['cur_dis_mean'] / amean[VTWO[1]]['cur_dis_mean'] - 1) * 100
_smin = min(float(FT[v]['soh'].min()) for v in VTWO); _rmin = min(float(RT[v]['soh'].min()) for v in VTWO)
_ylo = min(_smin, _rmin) - 1.5

fig = make_subplots(rows=3, cols=1, shared_xaxes=True, vertical_spacing=0.075,
                    subplot_titles=('SoH (%) — the outcome  (dots = raw daily, line = validated)',
                                    'Discharge-current magnitude (A) — why #1: heavier load',
                                    'Depth-of-discharge (%) — why #2: deeper cycling'))
for v in VTWO:
    g = FT[v]; rg = RT[v]; t = v[-6:]; c = CMAP[t]
    fig.add_scatter(x=rg['km'], y=rg['soh'], mode='markers', marker=dict(size=3, color=c), opacity=0.22,
                    legendgroup=t, showlegend=False, row=1, col=1,
                    hovertemplate=t + ' raw  %{x:.0f} km  %{y:.1f}%<extra></extra>')
    fig.add_scatter(x=g['km'], y=g['soh'], mode='lines+markers', line=dict(color=c, width=2.6),
                    marker=dict(size=4, color=c), name=t, legendgroup=t, row=1, col=1,
                    hovertemplate=t + '  %{x:.0f} km  %{y:.1f}%<extra></extra>')
    fig.add_scatter(x=[g['km'].iloc[-1]], y=[g['soh'].iloc[-1]], mode='text',
                    text=[f"  {g['soh'].iloc[-1]:.0f}%"], textposition='middle right',
                    textfont=dict(color=c, size=12), showlegend=False, row=1, col=1)
for r, (col, _, mag) in enumerate(EXPL, start=2):
    for i, v in enumerate(VTWO):
        g = FT[v]; t = v[-6:]; c = CMAP[t]
        y = pd.to_numeric(g[col], errors='coerce').abs() if mag else pd.to_numeric(g[col], errors='coerce')
        ys = y.rolling(3, center=True, min_periods=1).mean()
        fig.add_scatter(x=g['km'], y=y, mode='markers', marker=dict(size=3, color=c), opacity=0.22,
                        showlegend=False, row=r, col=1, hovertemplate=t + ' %{y:.2f}<extra></extra>')
        fig.add_scatter(x=g['km'], y=ys, mode='lines', line=dict(color=c, width=2.6),
                        legendgroup=t, showlegend=False, row=r, col=1)
        fig.add_hline(y=amean[v][col], line=dict(color=c, width=1, dash='dot'), opacity=0.7, row=r, col=1,
                      annotation_text=f'{t} avg {amean[v][col]:.1f}', annotation_position=('top left' if i == 0 else 'bottom left'),
                      annotation_font=dict(color=c, size=10))
for r in (1, 2, 3):
    fig.add_vline(x=CHK, line=dict(color='gray', width=1.2, dash='dash'), row=r, col=1)
fig.add_annotation(x=CHK, y=_ylo + 0.6, yref='y1', text=f'same distance: {CHK:,} km', showarrow=False,
                   xanchor='right', yanchor='bottom', font=dict(color='gray', size=10))
fig.update_yaxes(title_text='SoH %', range=[_ylo, 101], row=1, col=1)
fig.update_yaxes(title_text='current |A|', row=2, col=1)
fig.update_yaxes(title_text='DoD %', row=3, col=1)
fig.update_xaxes(range=[-800, CHK + 2600], row=3, col=1)
fig.update_xaxes(title_text='km driven (raw odometer)', row=3, col=1)
fig.update_layout(height=900, width=1040, template='plotly_white',
                  margin=dict(t=150, r=140, b=60, l=70),
                  title=dict(text='Same distance, different degradation: 217054 vs 217135<br>'
                                  f'<sub>Both driven {CHK:,} km, but 217054 ran {_pct:.0f}% heavier discharge current and cycled deeper -> '
                                  f'lost {drop[VTWO[0]]:.1f} pp vs {drop[VTWO[1]]:.1f} pp SoH</sub>',
                             x=0.02, xanchor='left', y=0.97, yanchor='top', font=dict(size=15)),
                  legend=dict(orientation='v', x=1.012, y=1, xanchor='left', yanchor='top', title='vehicle'))
fig.show()
print(f'Common distance {CHK:,} km:  217054 SoH {FT[VTWO[0]]["soh"].iloc[-1]:.0f}% (-{drop[VTWO[0]]:.1f}pp)  vs  '
      f'217135 SoH {FT[VTWO[1]]["soh"].iloc[-1]:.0f}% (-{drop[VTWO[1]]:.1f}pp)   | avg current '
      f'{amean[VTWO[0]]["cur_dis_mean"]:.1f} vs {amean[VTWO[1]]["cur_dis_mean"]:.1f} A (+{_pct:.0f}%), '
      f'DoD {amean[VTWO[0]]["dod_mean"]:.1f} vs {amean[VTWO[1]]["dod_mean"]:.1f}%')

## Does it generalize? Multiple same-distance vehicles

The 217054-vs-217135 pair suggested *heavier usage → more loss*. Tested across **6 non-flat, clean-odometer,
100%-start** vehicles at a common **25,000 km** (workflow-verified: 3 lenses + leave-one-out). **It does not
generalize.** At the same distance SoH still spans ~5 pp — but *which* vehicle degraded more is **not** explained
by usage: deeper DoD tracks *less* loss (ρ=−0.66), discharge current is null (ρ=−0.26). The only robust ordering
is **cumulative heat exposure** (months >40 °C, ρ=+0.88). Counterexamples: **217044** (highest DoD/current/
throughput, only −6.1 pp) and **217001** (lowest DoD, −10.3 pp). n=6 — indicative, not proof.

In [ ]:
from plotly.subplots import make_subplots
from scipy.stats import spearmanr

SET = ['217001', '217044', '217054', '217125', '217170', '217342']
D = 25000
fe_m = pd.read_parquet('data/euler/features/feature_table.parquet').sort_values(['vin', 'month'])

def vframe(t6):
    vin = [v for v in fe_m['vin'].unique() if v[-6:] == t6][0]
    g = fe_m[fe_m['vin'] == vin].sort_values('month').copy()
    odo = pd.to_numeric(g['odo_max'], errors='coerce').cummax()
    g['km'] = odo.diff().clip(lower=0, upper=8000).fillna(0).cumsum()
    g['soh_d'] = pd.to_numeric(g['soh'], errors='coerce')
    return g

DATA = {t: vframe(t) for t in SET}
rows = []
for t in SET:
    g = DATA[t]; m = g[g['km'] <= D]
    rows.append(dict(vin=t, drop=100 - float(np.interp(D, g['km'], g['soh_d'])),
                     dod=float(pd.to_numeric(m['dod_mean'], errors='coerce').mean()),
                     cur=float(pd.to_numeric(m['cur_dis_mean'], errors='coerce').abs().mean()),
                     hot=int((pd.to_numeric(m['temp_max'], errors='coerce') > 40).sum())))
S = pd.DataFrame(rows).sort_values('drop', ascending=False).reset_index(drop=True)
COL = {t: PAL[i] for i, t in enumerate(S['vin'])}
r_dod = spearmanr(S['dod'], S['drop']).correlation
r_cur = spearmanr(S['cur'], S['drop']).correlation
r_hot = spearmanr(S['hot'], S['drop']).correlation

fig = make_subplots(rows=2, cols=3, specs=[[{'colspan': 3}, None, None], [{}, {}, {}]],
                    row_heights=[0.56, 0.44], vertical_spacing=0.14, horizontal_spacing=0.07,
                    subplot_titles=(f'SoH vs distance — 6 non-flat vehicles (compared at {D:,} km)',
                                    f'mean DoD vs drop   (ρ={r_dod:+.2f})  ✗ wrong way',
                                    f'mean discharge |A| vs drop   (ρ={r_cur:+.2f})  ✗ null',
                                    f'hot months >40°C vs drop   (ρ={r_hot:+.2f})  ✓ orders it'))
for t in S['vin']:
    g = DATA[t]; m = g[g['km'] <= D]; c = COL[t]
    fig.add_scatter(x=m['km'], y=m['soh_d'], mode='lines+markers', line=dict(color=c, width=2),
                    marker=dict(size=3, color=c), name=t, legendgroup=t, row=1, col=1,
                    hovertemplate=t + '  %{x:.0f} km  %{y:.1f}%<extra></extra>')
    fig.add_scatter(x=[m['km'].iloc[-1] if m['km'].iloc[-1] <= D else D], y=[float(np.interp(D, g['km'], g['soh_d']))],
                    mode='text', text=[f"  {t}"], textposition='middle right', textfont=dict(color=c, size=9),
                    showlegend=False, row=1, col=1)
fig.add_vline(x=D, line=dict(color='gray', width=1.2, dash='dash'), row=1, col=1)
_lo = float(min(np.interp(D, DATA[t]['km'], DATA[t]['soh_d']) for t in SET))
fig.add_annotation(x=D, y=_lo - 0.6, yref='y1', text=f'at {D:,} km: {_lo:.1f}–{S["drop"].min()*0 + 100 - S["drop"].min():.1f}% SoH',
                   showarrow=False, xanchor='right', yanchor='top', font=dict(color='gray', size=10))
for ci, (feat, xlab) in enumerate([('dod', 'mean DoD (%)'), ('cur', 'mean discharge |A|'), ('hot', 'hot months >40°C')], start=1):
    for t in S['vin']:
        rr = S[S['vin'] == t].iloc[0]
        fig.add_scatter(x=[rr[feat]], y=[rr['drop']], mode='markers', marker=dict(size=12, color=COL[t]),
                        legendgroup=t, showlegend=False, row=2, col=ci,
                        hovertemplate=t + f'  {feat}=%{{x}}  drop=%{{y:.1f}}pp<extra></extra>')
    xv = S[feat].to_numpy(float); yv = S['drop'].to_numpy(float)
    mm, bb = np.polyfit(xv, yv, 1); xs = np.array([xv.min(), xv.max()])
    fig.add_scatter(x=xs, y=mm * xs + bb, mode='lines', line=dict(color='gray', width=1, dash='dot'),
                    showlegend=False, row=2, col=ci)
    fig.update_xaxes(title_text=xlab, row=2, col=ci)
fig.update_yaxes(title_text='SoH drop (pp)', row=2, col=1)
# annotate the two counterexamples in the DoD panel
for t, note in [('217044', '217044: max stress,<br>low drop'), ('217001', '217001: low DoD,<br>high drop')]:
    rr = S[S['vin'] == t].iloc[0]
    fig.add_annotation(x=rr['dod'], y=rr['drop'], text=note, row=2, col=1, showarrow=True, arrowhead=2,
                       arrowsize=0.7, ax=18, ay=(-22 if t == '217044' else 22), font=dict(color=COL[t], size=9))
fig.update_yaxes(title_text='SoH %', range=[88, 101], row=1, col=1)
fig.update_xaxes(title_text='km driven (raw odometer)', row=1, col=1)
fig.update_layout(height=820, width=1080, template='plotly_white', margin=dict(t=150, r=60),
                  title=dict(text='Same distance, different degradation — across 6 non-flat vehicles<br>'
                                  '<sub>At 25,000 km SoH spans ~5 pp, but harder usage does NOT explain which aged more '
                                  '(DoD ρ=-0.66, current ρ=-0.26); only cumulative heat exposure orders it (ρ=+0.88, n=6 - indicative)</sub>',
                             x=0.02, xanchor='left', y=0.975, yanchor='top', font=dict(size=15)),
                  legend=dict(orientation='h', y=1.075, x=0, font=dict(size=10), title='vehicle'))
fig.show()
print('Same distance (25,000 km), 6 non-flat vehicles — ordered by SoH drop:')
print(S.round(1).to_string(index=False))
print(f'\nSpearman vs SoH drop:  DoD {r_dod:+.2f} (wrong way) · discharge|A| {r_cur:+.2f} (null) · hot-months>40C {r_hot:+.2f} (only robust one)')

## Mahindra — odometer logged too late (omitted)

In [ ]:
print('Mahindra omitted - NOT ONE Mahindra vehicle has its odometer logged from a fresh (100% SoH) state.')
print("The odometer only first appears after heavy use - e.g. L22039's first reading is 23,352 km at 78% SoH -")
print('so a trustworthy SoH-vs-km curve from a 100% baseline cannot be built for Mahindra. (Euler above is clean.)')

## Conclusion (clean odometer)

With a trustworthy km axis (odometer from a fresh 100% start), Euler vehicles driven the **same long distance**
still differ in SoH by ~10-15 pp. Distance does not set degradation - usage conditions (cycling/throughput,
depth-of-discharge, temperature) do. Mahindra can't be assessed this way: its odometer telemetry starts too late.